# 08 — Databases & ORMs

Every Node.js backend talks to a database. Understanding database interaction patterns is essential for interviews.

---

## Table of Contents
1. SQL vs NoSQL
2. MongoDB & Mongoose
3. SQL Databases & Sequelize/Prisma
4. Connection Pooling
5. Transactions
6. Data Modeling Patterns
7. Indexing
8. Interview Questions

---
## 1. SQL vs NoSQL

| Feature | SQL (PostgreSQL, MySQL) | NoSQL (MongoDB, Redis) |
|---------|----------------------|----------------------|
| Data Model | Tables with rows & columns | Documents, Key-Value, Graph |
| Schema | Fixed schema (migrations) | Flexible / schema-less |
| Relationships | JOINs (normalized data) | Embedded docs / refs (denormalized) |
| Transactions | Full ACID | Limited (MongoDB 4.0+ has multi-doc) |
| Scaling | Vertical (read replicas) | Horizontal (sharding) |
| Query Language | SQL | Database-specific |
| Best For | Complex relations, transactions | Flexible schemas, rapid iteration |

### When to choose what:
- **SQL** — Financial data, e-commerce, complex relationships, reporting
- **MongoDB** — Content management, real-time analytics, rapid prototyping
- **Redis** — Caching, sessions, real-time leaderboards, pub/sub
- **Neo4j** — Social networks, recommendation engines

> **Interview Tip:** Don't say one is "better" than the other. Show you understand the trade-offs and can choose based on requirements.

---
## 2. MongoDB & Mongoose

Mongoose is an ODM (Object Document Mapper) for MongoDB — it adds schema validation, middleware, and helper methods.

```javascript
const mongoose = require('mongoose');

// Connection
await mongoose.connect('mongodb://localhost:27017/myapp', {
    useNewUrlParser: true,
    useUnifiedTopology: true,
});

// Schema definition
const userSchema = new mongoose.Schema({
    name:      { type: String, required: true, trim: true },
    email:     { type: String, required: true, unique: true, lowercase: true },
    password:  { type: String, required: true, minlength: 8, select: false },
    role:      { type: String, enum: ['user', 'admin'], default: 'user' },
    age:       { type: Number, min: 18 },
    posts:     [{ type: mongoose.Schema.Types.ObjectId, ref: 'Post' }],
    createdAt: { type: Date, default: Date.now },
});

// Indexes
userSchema.index({ email: 1 });
userSchema.index({ name: 'text' }); // Text search index

// Middleware (hooks)
userSchema.pre('save', async function(next) {
    if (this.isModified('password')) {
        this.password = await bcrypt.hash(this.password, 12);
    }
    next();
});

// Instance methods
userSchema.methods.checkPassword = async function(candidatePassword) {
    return bcrypt.compare(candidatePassword, this.password);
};

// Static methods
userSchema.statics.findByEmail = function(email) {
    return this.findOne({ email });
};

// Virtual properties (not stored in DB)
userSchema.virtual('profileUrl').get(function() {
    return `/users/${this._id}`;
});

const User = mongoose.model('User', userSchema);
```

In [ ]:
// Mongoose CRUD operations (conceptual — no live DB needed)

const operations = {
    'CREATE': {
        single: "await User.create({ name: 'Alice', email: 'alice@ex.com' })",
        multiple: "await User.insertMany([{ name: 'Bob' }, { name: 'Charlie' }])",
    },
    'READ': {
        findAll: "await User.find({})",
        findOne: "await User.findById(id)",
        withFilter: "await User.find({ role: 'admin' }).sort('-createdAt').limit(10)",
        withPopulate: "await User.findById(id).populate('posts')",
        selectFields: "await User.find({}).select('name email -_id')",
    },
    'UPDATE': {
        findAndUpdate: "await User.findByIdAndUpdate(id, { name: 'New' }, { new: true, runValidators: true })",
        updateMany: "await User.updateMany({ role: 'user' }, { $set: { active: true } })",
    },
    'DELETE': {
        findAndDelete: "await User.findByIdAndDelete(id)",
        deleteMany: "await User.deleteMany({ active: false })",
    }
};

for (const [op, methods] of Object.entries(operations)) {
    console.log(`\n=== ${op} ===`);
    for (const [name, code] of Object.entries(methods)) {
        console.log(`  ${name}: ${code}`);
    }
}

### MongoDB Aggregation Pipeline:
```javascript
// Powerful data processing pipeline
const stats = await Order.aggregate([
    { $match: { status: 'completed' } },        // Filter
    { $group: {                                   // Group by
        _id: '$userId',
        totalSpent: { $sum: '$amount' },
        orderCount: { $sum: 1 },
        avgOrder: { $avg: '$amount' },
    }},
    { $sort: { totalSpent: -1 } },               // Sort descending
    { $limit: 10 },                               // Top 10
    { $lookup: {                                   // JOIN with users
        from: 'users',
        localField: '_id',
        foreignField: '_id',
        as: 'user'
    }},
    { $unwind: '$user' },                         // Flatten array
    { $project: {                                  // Select fields
        userName: '$user.name',
        totalSpent: 1,
        orderCount: 1
    }}
]);
```

---
## 3. SQL Databases & ORMs

### Popular Node.js SQL tools:
- **Sequelize** — Full ORM, mature, supports many SQL dialects
- **Prisma** — Modern ORM, TypeScript-first, schema-driven
- **Knex.js** — Query builder (not full ORM), flexible
- **TypeORM** — Decorator-based, good with TypeScript/NestJS

### Prisma example (increasingly popular in interviews):
```prisma
// schema.prisma
model User {
    id        Int      @id @default(autoincrement())
    email     String   @unique
    name      String
    posts     Post[]
    createdAt DateTime @default(now())
}

model Post {
    id        Int      @id @default(autoincrement())
    title     String
    content   String?
    author    User     @relation(fields: [authorId], references: [id])
    authorId  Int
}
```

```javascript
const { PrismaClient } = require('@prisma/client');
const prisma = new PrismaClient();

// Create with relation
const user = await prisma.user.create({
    data: {
        name: 'Alice',
        email: 'alice@example.com',
        posts: {
            create: { title: 'My first post', content: 'Hello!' }
        }
    },
    include: { posts: true } // Include related posts in response
});
```

---
## 4. Connection Pooling

Opening a DB connection is expensive (~20-50ms). Connection pooling reuses connections.

```
Without pooling:
  Request 1 → Open connection → Query → Close connection
  Request 2 → Open connection → Query → Close connection
  (50ms overhead each time)

With pooling:
  Request 1 → Get connection from pool → Query → Return to pool
  Request 2 → Get connection from pool → Query → Return to pool
  (connections are reused, ~0ms overhead)
```

```javascript
// PostgreSQL with pg library
const { Pool } = require('pg');
const pool = new Pool({
    host: 'localhost',
    port: 5432,
    database: 'myapp',
    user: 'admin',
    password: 'secret',
    max: 20,                // Max connections in pool
    idleTimeoutMillis: 30000, // Close idle connections after 30s
    connectionTimeoutMillis: 2000, // Timeout waiting for connection
});

// Query using pool (automatically gets/returns connection)
const result = await pool.query('SELECT * FROM users WHERE id = $1', [userId]);
```

### Pool size formula:
> `pool_size = (core_count * 2) + effective_spindle_count`  
> For a 4-core server with SSD: `(4 * 2) + 1 = 9` connections

Too many connections: wasted memory, context switching  
Too few connections: requests queue up, latency increases

---
## 5. Transactions

Transactions ensure **ACID** properties: Atomicity, Consistency, Isolation, Durability.

### When to use:
- Transferring money between accounts
- Creating an order with multiple line items
- Any operation where partial completion is worse than complete failure

```javascript
// SQL transaction (PostgreSQL)
const client = await pool.connect();
try {
    await client.query('BEGIN');
    await client.query('UPDATE accounts SET balance = balance - $1 WHERE id = $2', [100, fromId]);
    await client.query('UPDATE accounts SET balance = balance + $1 WHERE id = $2', [100, toId]);
    await client.query('COMMIT');
} catch (err) {
    await client.query('ROLLBACK');
    throw err;
} finally {
    client.release();
}

// MongoDB transaction (v4.0+)
const session = await mongoose.startSession();
try {
    session.startTransaction();
    await Account.updateOne({ _id: fromId }, { $inc: { balance: -100 } }, { session });
    await Account.updateOne({ _id: toId }, { $inc: { balance: 100 } }, { session });
    await session.commitTransaction();
} catch (err) {
    await session.abortTransaction();
    throw err;
} finally {
    session.endSession();
}
```

---
## 6. Data Modeling Patterns

### MongoDB — Embedding vs Referencing:

| Pattern | When to Use | Example |
|---------|------------|--------|
| **Embedding** | Data always accessed together, 1:few relationship | User with addresses |
| **Referencing** | Data accessed independently, 1:many/many:many | User with 10000 posts |

```javascript
// Embedding (denormalized)
const orderSchema = {
    customer: { name: 'Alice', email: 'alice@ex.com' }, // embedded!
    items: [
        { product: 'Widget', price: 29.99, quantity: 2 }
    ]
};

// Referencing (normalized)
const orderSchema = {
    customer: ObjectId('...'), // reference to User collection
    items: [
        { product: ObjectId('...'), quantity: 2 } // reference to Product
    ]
};
```

### SQL — Normalization:
- **1NF** — No repeating groups (each cell = one value)
- **2NF** — 1NF + no partial dependencies (all non-key columns depend on the whole key)
- **3NF** — 2NF + no transitive dependencies (non-key columns don't depend on other non-key columns)

---
## 7. Indexing

Indexes make queries fast but slow down writes (because the index must be updated).

### When to index:
- Fields used in `WHERE` / `find()` filters
- Fields used in `ORDER BY` / `sort()`
- Fields used in `JOIN ON` / `$lookup`
- Foreign keys

### When NOT to index:
- Small collections (full scan is fast enough)
- Fields that are rarely queried
- Fields with very low cardinality (e.g., boolean)
- Write-heavy collections (indexes slow writes)

```javascript
// MongoDB indexes
userSchema.index({ email: 1 });                    // Single field
userSchema.index({ lastName: 1, firstName: 1 });  // Compound
userSchema.index({ email: 1 }, { unique: true }); // Unique
userSchema.index({ bio: 'text' });                 // Text search
userSchema.index({ location: '2dsphere' });        // Geospatial
userSchema.index({ createdAt: 1 }, { expireAfterSeconds: 86400 }); // TTL
```

---
## 8. Interview Questions & Answers

### Q1: When would you choose MongoDB over PostgreSQL?
**A:** MongoDB when: schema is evolving rapidly, data is document-shaped (nested/hierarchical), you need horizontal scaling, or for real-time analytics. PostgreSQL when: data has complex relationships, you need ACID transactions, strong consistency, or complex queries with JOINs.

### Q2: What is connection pooling and why is it important?
**A:** Connection pooling maintains a set of reusable database connections. Creating a new connection takes 20-50ms — with pooling, this overhead is avoided. The pool manages connection lifecycle (creation, reuse, cleanup). Without it, a high-traffic server would exhaust database connections.

### Q3: Explain ACID properties.
**A:** Atomicity: all operations succeed or all fail. Consistency: data always moves from one valid state to another. Isolation: concurrent transactions don't interfere. Durability: committed data survives crashes. SQL databases fully support ACID; MongoDB added multi-document transactions in v4.0.

### Q4: What is the N+1 query problem?
**A:** When you fetch N records, then make 1 additional query for each record to get related data (N+1 total queries). Fix: use eager loading (`populate()` in Mongoose, `include` in Sequelize/Prisma, JOIN in SQL) to fetch all related data in a single query.

### Q5: Embedding vs Referencing in MongoDB — how do you decide?
**A:** Embed when: data is always accessed together, the subdocument is small and bounded (addresses, preferences). Reference when: data is accessed independently, the array could grow unbounded (comments, posts), or many-to-many relationships exist.

### Q6: How do you handle database migrations?
**A:** SQL: use migration tools (Prisma Migrate, Sequelize CLI, Knex migrations) that track schema changes as versioned files. MongoDB: use migration scripts or libraries like `migrate-mongo`. Always test migrations in staging, have rollback plans, and never run schema changes during peak traffic.